In [ ]:
!pip install --quiet google-play-scraper pysentimiento

In [ ]:
import requests
import time
import pandas as pd
from typing import List, Dict, Any
from google_play_scraper import Sort, reviews, app as play_app

# Extracción de la Play Store

In [ ]:
def extract_playstore_reviews(app_id: str, target_count: int = 5000) -> pd.DataFrame:
    all_reviews: List[Dict[str, Any]] = []
    continuation_token = None

    print(f"Iniciando extracción detallada para {app_id}...")

    while len(all_reviews) < target_count:
        result, continuation_token = reviews(
            app_id,
            lang='es',
            country='mx',
            sort=Sort.NEWEST,
            count=1000,
            continuation_token=continuation_token
        )

        all_reviews.extend(result)
        print(f"Volumen extraído: {len(all_reviews)} reseñas.")

        if not continuation_token:
            break

        time.sleep(1)

    df = pd.DataFrame(all_reviews[:target_count])

    # Estandarización del esquema base y traducción de metadata exclusiva
    df.rename(columns={
        'reviewId': 'id_resena',
        'userName': 'usuario',
        'content': 'texto',
        'score': 'estrellas',
        'at': 'fecha',
        'reviewCreatedVersion': 'version_app',
        'userImage': 'imagen_usuario',
        'thumbsUpCount': 'votos_utilidad',
        'replyContent': 'texto_respuesta',
        'repliedAt': 'fecha_respuesta'
    }, inplace=True)

    # Eliminamos 'appVersion' para evitar redundancia, ya que 'version_app' capturó la versión al momento de la reseña
    if 'appVersion' in df.columns:
        df.drop(columns=['appVersion'], inplace=True)

    # Inyectar la columna 'titulo' como nula para coincidir con el esquema de App Store
    df['titulo'] = None

    return df

In [ ]:
# Package ID oficial de BBVA México en Google Play
app_package = 'com.bancomer.mbanking'

df_playstore = extract_playstore_reviews(app_id=app_package, target_count=5000)

In [ ]:
output_path = 'bbva_playstore_sample.csv'
df_playstore.to_csv(output_path, index=False, encoding='utf-8')
print(f"Proceso finalizado. Dataset guardado en: {output_path}")

In [ ]:
df = df_playstore

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Paleta institucional UP
UP_PALETTE = {'Azul': '#002D72', 'Verde': '#215347', 'Vino': '#7A1A37', 'Dorado': '#B9975B'}

# 1. Cálculo de longitud de palabras
df['conteo_palabras'] = df['texto'].apply(lambda x: len(str(x).split()))

# 2. Gráfico de distribución de longitud (Detección de colas pesadas)
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='conteo_palabras', bins=50, kde=True, color=UP_PALETTE['Verde'])
plt.axvline(df['conteo_palabras'].median(), color=UP_PALETTE['Dorado'], linestyle='--', label=f"Mediana: {df['conteo_palabras'].median()} palabras")
plt.title('Distribución de Longitud de Reseñas (Detección de Colas Pesadas)', fontsize=14, color=UP_PALETTE['Azul'], fontweight='bold')
plt.xlabel('Número de Palabras', fontsize=12)
plt.ylabel('Frecuencia', fontsize=12)
plt.xlim(0, 100) # Acotado para visualizar el cuerpo principal
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 3. Pipeline de limpieza ajustado a textos ultracortos
def limpiar_texto_bancario(texto):
    texto = str(texto).lower()
    # Quitar URLs y menciones, pero preservando signos de puntuación clave y emojis
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)
    texto = re.sub(r'\@\w+|\#', '', texto)
    # Reducir caracteres repetidos excesivos sin destruir la palabra (ej. "noooo sirveee" -> "no sirve")
    texto = re.sub(r'(.)\1{2,}', r'\1\1', texto)
    return texto.strip()

# Filtrar: Reducimos el umbral inferior a 2 palabras y el superior a 60 (para cortar el spam)
df_curado = df[(df['conteo_palabras'] >= 2) & (df['conteo_palabras'] <= 60)].copy()
df_curado['texto_limpio'] = df_curado['texto'].apply(limpiar_texto_bancario)

print(f"Retención del corpus ajustado: {len(df_curado)} reseñas ({(len(df_curado)/len(df))*100:.1f}%)")

In [ ]:
from pysentimiento import create_analyzer

# Inicializar el analizador basado en RoBERTa (robertuito-sentiment-analysis)
analyzer = create_analyzer(task="sentiment", lang="es")

# Función para inferencia en batch
def predecir_sentimiento_hf(texto):
    output = "NEU"
    try:
        # Devuelve POS, NEG o NEU
        output = analyzer.predict(texto).output
        print(texto, output)
        return output
    except:
        print(texto, output)
        return "NEU"

# Advertencia: Esto puede tomar unos minutos dependiendo de tu CPU/GPU
df_curado['sentimiento_roberta'] = df_curado['texto_limpio'].apply(predecir_sentimiento_hf)

# Mapeo para gráfica
df_curado['sentimiento_roberta_2'] = df_curado['sentimiento_roberta'].map({'NEG': 'Negativo', 'NEU': 'Neutral', 'POS': 'Positivo'})

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de paleta institucional UP
UP_PALETTE = {
    'Azul': '#002D72',
    'Verde': '#215347',
    'Vino': '#7A1A37',
    'Dorado': '#B9975B'
}
sns.set_theme(style="whitegrid")

# 1. Gráfico de distribución de estrellas
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df_curado, x='estrellas', palette=[UP_PALETTE['Vino'], UP_PALETTE['Dorado'], UP_PALETTE['Verde'], UP_PALETTE['Azul'], UP_PALETTE['Azul']])
plt.title('Distribución de Calificaciones por Estrellas', fontsize=14, color=UP_PALETTE['Azul'], fontweight='bold')
plt.xlabel('Estrellas Otorgadas', fontsize=12)
plt.ylabel('Volumen de Reseñas', fontsize=12)
plt.show()

In [ ]:
from textblob import TextBlob

# Análisis de polaridad léxica base
df_curado['polaridad_texto'] = df_curado['texto'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)

# Categorización del sentimiento
def categorizar_sentimiento(score):
    if score < -0.1: return 'Negativo'
    elif score > 0.1: return 'Positivo'
    else: return 'Neutral'

df_curado['sentimiento_real_textblob'] = df_curado['polaridad_texto'].apply(categorizar_sentimiento)

# 2. Gráfico de barras apiladas: Incongruencia texto vs estrellas
crosstab_data = pd.crosstab(df_curado['estrellas'], df_curado['sentimiento_real_textblob'], normalize='index') * 100
crosstab_data = crosstab_data[['Negativo', 'Neutral', 'Positivo']]

crosstab_data.plot(kind='bar', stacked=True, figsize=(10, 6),
                   color=[UP_PALETTE['Vino'], '#EAEAEA', UP_PALETTE['Verde']])
plt.title('Sentimiento (Textblob) vs. Calificación de estrellas', fontsize=14, color=UP_PALETTE['Azul'], fontweight='bold')
plt.xlabel('Estrellas Otorgadas', fontsize=12)
plt.ylabel('Porcentaje (%)', fontsize=12)
plt.legend(title='Sentimiento del Texto', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico de barras apiladas: Incongruencia texto vs estrellas con RoBERTa
crosstab_hf = pd.crosstab(df_curado['estrellas'], df_curado['sentimiento_roberta_2'], normalize='index') * 100
crosstab_hf = crosstab_hf[['Negativo', 'Neutral', 'Positivo']]

crosstab_hf.plot(kind='bar', stacked=True, figsize=(10, 6),
                 color=[UP_PALETTE['Vino'], '#EAEAEA', UP_PALETTE['Verde']])
plt.title('Sentimiento (RoBERTa) vs Calificación de Estrellas', fontsize=14, color=UP_PALETTE['Azul'], fontweight='bold')
plt.xlabel('Estrellas Otorgadas', fontsize=12)
plt.ylabel('Porcentaje (%)', fontsize=12)
plt.legend(title='Sentimiento del Texto', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Identificar el Top 8 de versiones con mayor volumen de reseñas
top_8_versiones = df_curado['version_app'].value_counts().nlargest(8).index
df_top_versiones = df_curado[df_curado['version_app'].isin(top_8_versiones)].copy()

# 2. Ordenar lógicamente solo las versiones del Top 8 (Semantic Versioning)
versiones_ordenadas = sorted(
    top_8_versiones,
    key=lambda x: [int(i) if i.isdigit() else i for i in str(x).split('.')]
)

# 3. Filtrar por el sentimiento negativo extraído con RoBERTa
df_negativo = df_top_versiones[df_top_versiones['sentimiento_roberta_2'] == 'Negativo']

# 4. Paleta categórica de la Universidad Panamericana (8 tonos de alto contraste)
UP_8_COLORS = [
    '#002D72', # Azul oscuro
    '#B9975B', # Dorado
    '#7A1A37', # Vino
    '#215347', # Verde
    '#4B74A3', # Azul claro
    '#D4BE96', # Dorado claro
    '#A35A6F', # Vino claro
    '#568779'  # Verde claro
]
paleta_versiones = sns.color_palette(UP_8_COLORS)

# 5. Generación del gráfico
plt.figure(figsize=(14, 7))
ax = sns.histplot(
    data=df_negativo,
    x='fecha',
    hue='version_app',
    hue_order=versiones_ordenadas,
    multiple="stack",
    palette=paleta_versiones[:len(versiones_ordenadas)],
    bins=30,
    edgecolor='white',
    linewidth=0.5
)

plt.title('Volumen reviews negativos (RoBERTa) por Versión de App (Top 8)', fontsize=14, color='#002D72', fontweight='bold')
plt.xlabel('Fecha de Publicación', fontsize=12)
plt.ylabel('Frecuencia de Quejas', fontsize=12)
plt.xticks(rotation=45)

# Desplazar la leyenda para no obstruir los datos
sns.move_legend(ax, "upper left", bbox_to_anchor=(1.01, 1), title='Versión de la App')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Conservar solo reseñas con interacción
df_votos = df_curado[df_curado['votos_utilidad'] > 0].copy()

# 3. Filtrar outliers
df_sin_outliers = df_votos

# 4. Generar el gráfico de violín
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=df_sin_outliers,
    x='sentimiento_roberta_2',
    y='votos_utilidad',
    palette=[UP_PALETTE['Vino'], '#EAEAEA', UP_PALETTE['Verde']],
    order=['Negativo', 'Neutral', 'Positivo'],
    inner='quartile', # Dibuja los cuartiles como líneas punteadas dentro del área
    cut=0,            # Fuerza a que el gráfico termine exactamente en los datos reales (no extrapola)
    density_norm='area' # Escala el ancho de cada violín por el número de observaciones
)

plt.title('Densidad de Votos de Utilidad por Sentimiento (RoBERTa)', fontsize=14, color=UP_PALETTE['Azul'], fontweight='bold')
plt.xlabel('Sentimiento Extraído', fontsize=12)
plt.ylabel('Cantidad de Votos de Utilidad', fontsize=12)
plt.tight_layout()
plt.yscale('log')
plt.show()

In [ ]:
from google_play_scraper import app

In [ ]:
app(
    'com.bancomer.mbanking',
    lang='es', # defaults to 'en'
    country='mx' # defaults to 'us'
)["reviews"]

In [ ]:
app_package = 'com.nu.production'
app(
    app_package,
    lang='es', # defaults to 'en'
    country='mx' # defaults to 'us'
)["reviews"]

In [ ]:
app_package = 'mx.bancosantander.supermovil'
app(
    app_package,
    lang='es', # defaults to 'en'
    country='mx' # defaults to 'us'
)["reviews"]

In [ ]:
app_package = 'mx.com.bancoazteca.bazdigitalmovil'
app(
    app_package,
    lang='es', # defaults to 'en'
    country='mx' # defaults to 'us'
)["reviews"]

In [ ]:
app_package = 'org.microemu.android.model.common.VTUserApplicationBNRTMB'
app(
    app_package,
    lang='es', # defaults to 'en'
    country='mx' # defaults to 'us'
)["reviews"]